In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/datasets/phhasian0710/seed-iv/Channel Order.xlsx
/kaggle/input/datasets/phhasian0710/seed-iv/ReadMe.txt
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/4_20151118.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/14_20151208.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/5_20160413.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/3_20151018.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/6_20150511.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/15_20150514.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/10_20151021.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/12_20150804.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/13_20151125.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/11_20150921.mat
/kaggle/input/datasets/phhasian0710/seed-iv/eye_feature_smooth/2/8_20151110.

In [ ]:
!pip install mne PyWavelets scikit-learn tqdm matplotlib seaborn pandas scipy numpy torch torchvision torchaudio imbalanced-learn

In [3]:
# =============================================================
# BALANCED SOGNN FOR SEED-IV EEG EMOTION RECOGNITION
# Enhanced with Class Balance Optimization
# Fixed version: Resolves torch.load UnpicklingError
# =============================================================
#
# OVERVIEW OF WHAT THIS SCRIPT DOES:
# ─────────────────────────────────────────────────────────────
# This pipeline classifies EEG signals into 4 emotion categories:
#   0=Neutral, 1=Sad, 2=Fear, 3=Happy
#
# DATASET: SEED-IV (15 subjects, 3 sessions each, 24 trials/session)
#
# PIPELINE STEPS:
#   1. Load .mat EEG feature files → extract DE (Differential Entropy) features
#   2. Standardize time length across all trials
#   3. Normalize per-subject (removes subject-specific signal bias)
#   4. Extract multi-features: DE, PSD, Sample Entropy, Permutation Entropy
#   5. LOSO (Leave-One-Subject-Out) cross-validation:
#        - Train on 14 subjects, validate on part of those, test on held-out subject
#        - Uses balanced sampling, focal loss, contrastive learning
#   6. Evaluate: accuracy, macro F1, per-class accuracy, confusion matrix
#
# MODEL: SOGNN (Self-Organized Graph Neural Network)
#   - CNN blocks extract local temporal EEG features
#   - Graph convolution captures spatial channel relationships
#   - Region-aware processor groups channels by brain region
#   - Multi-scale temporal module captures patterns at different time scales
# =============================================================

# ─── CELL 0: INSTALL DEPENDENCIES ───────────────────────────
# Run this only once in Kaggle/Colab to install required libraries
# !pip install mne PyWavelets scikit-learn tqdm matplotlib seaborn pandas scipy numpy torch torchvision torchaudio imbalanced-learn

# ─── CELL 1: IMPORTS AND CONFIGURATION ──────────────────────

import os
import glob
import time
import math
import pickle
import random
from collections import Counter
from itertools import permutations

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.stats import entropy, kurtosis, zscore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── FIX: Allow numpy globals when loading checkpoints with torch.load ──────
# PyTorch 2.6+ changed torch.load default to weights_only=True, which blocks
# numpy types saved inside checkpoint dicts (e.g. per_class_val as np.ndarray).
#
# The checkpoint saved by BalancedTrainer stores 'per_class_val' as a numpy array.
# PyTorch's safe unpickling blocks numpy types by default. We must allowlist:
#   - numpy.ndarray          : the actual array class
#   - numpy._core.multiarray._reconstruct : internal function used to rebuild arrays
#   - numpy.dtype            : dtype metadata attached to every ndarray
#   - numpy.core.multiarray._reconstruct  : older alias (numpy < 2.0 compatibility)
#
# This is safe because WE wrote the checkpoint ourselves during training.
# Only do this for checkpoints from trusted sources (your own training runs).
import numpy._core.multiarray
import numpy as _np_fix
torch.serialization.add_safe_globals([
    _np_fix.ndarray,                           # numpy array class itself
    _np_fix.dtype,                             # dtype objects stored inside arrays
    _np_fix._core.multiarray._reconstruct,     # numpy 2.x reconstruction function
]) 

import warnings
warnings.filterwarnings('ignore')


# ─── SEED FUNCTION ───────────────────────────────────────────
def set_seed(seed=42):
    """
    Sets all random seeds for reproducibility across:
    - Python's built-in random module
    - NumPy
    - PyTorch (CPU and GPU)
    Also disables non-deterministic CUDA ops.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✓ Seed set to {seed}")


# ─── CONFIGURATION ───────────────────────────────────────────
class Config:
    """
    Central configuration class. All hyperparameters, paths, and feature
    settings are stored here. Modify values here to tune the pipeline.

    KEY DESIGN DECISIONS:
    - Focal Loss: Penalizes easy examples more, forces model to learn hard ones
    - Balanced Sampling: Ensures each mini-batch has equal class representation
    - Contrastive Learning: Pulls same-class embeddings together in feature space
    - Label Smoothing: Prevents the model from being overconfident (avoids NaN loss)
    """

    # ── Paths ─────────────────────────────────────────────────
    DATA_PATH = '/kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth/'
    OUTPUT_PATH = './processed_data/'
    MODEL_PATH = './saved_models/'
    RESULTS_PATH = './results/'

    # ── Dataset Parameters ────────────────────────────────────
    NUM_SUBJECTS = 15        # 15 subjects total in SEED-IV
    NUM_CLASSES = 4          # Neutral, Sad, Fear, Happy
    NUM_CHANNELS = 62        # 62 EEG electrodes
    NUM_SESSIONS = 3         # 3 recording sessions per subject
    TRIALS_PER_SESSION = 24  # 24 emotion-labeled trials per session

    # Emotion label for each of the 24 trials (same order for all sessions)
    # 0=Neutral, 1=Sad, 2=Fear, 3=Happy
    TRIAL_EMOTION_MAPPING = [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1,
                             1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3]
    EMOTION_NAMES = {0: 'Neutral', 1: 'Sad', 2: 'Fear', 3: 'Happy'}

    # ── Feature Extraction ────────────────────────────────────
    # We compute 4 feature types, each across 5 frequency bands (delta, theta,
    # alpha, beta, gamma) → 4 × 5 = 20 features per channel per time step
    FEATURE_TYPES = ['DE', 'PSD', 'SE', 'PE']
    NUM_FREQ_BANDS = 5
    TOTAL_FEATURES = len(FEATURE_TYPES) * NUM_FREQ_BANDS  # = 20

    # ── Temporal Parameters ───────────────────────────────────
    # All trials are padded/cropped to a fixed 35 time steps for consistent batching
    TARGET_TIME_LENGTH = 35

    # ── Class Balance Settings ────────────────────────────────
    # CLASS_WEIGHTS: higher weight → loss penalizes misclassification of that class more
    # Neutral=0.8 and Fear=0.7 are given less weight (easier to classify);
    # Happy=1.2 gets more weight (harder or under-represented)
    CLASS_WEIGHTS = [0.8, 1.0, 0.7, 1.2]

    # Balanced sampler draws SAMPLES_PER_CLASS samples from each class per batch
    USE_BALANCED_SAMPLING = True
    SAMPLES_PER_CLASS = 32             # 32 samples × 4 classes = 128 per batch
    BATCH_SIZE = 128

    # ── Focal Loss ────────────────────────────────────────────
    # Focal Loss: FL(p_t) = -α_t × (1 - p_t)^γ × log(p_t)
    # γ (gamma): focuses on hard examples; higher γ → more focus on misclassified
    # α (alpha): per-class weight to handle imbalance further
    USE_FOCAL_LOSS = True
    FOCAL_GAMMA = 2.5
    FOCAL_ALPHA = [0.2, 0.25, 0.2, 0.35]   # More weight on Happy and Sad
    LABEL_SMOOTHING = 0.1                    # Soft labels: 0.9 for true class, 0.1/3 for others

    # ── Data Augmentation ─────────────────────────────────────
    # Applied only during training to prevent overfitting
    USE_AUGMENTATION = True
    AUGMENTATION_TYPES = ['noise', 'mixup', 'cutmix', 'time_warp']
    NOISE_STD = 0.02      # Gaussian noise standard deviation
    MIXUP_ALPHA = 0.3     # Beta distribution parameter for Mixup interpolation
    CUTMIX_PROB = 0.5     # Probability of applying CutMix

    # ── Contrastive Learning ──────────────────────────────────
    # NT-Xent loss: pulls same-class embeddings closer in feature space
    # Acts as a regularizer alongside the main classification loss
    USE_CONTRASTIVE = True
    CONTRASTIVE_TEMPERATURE = 0.3    # Lower temp = sharper similarity distribution
    CONTRASTIVE_WEIGHT = 0.2         # Multiplied with contrastive loss before adding to cls loss

    # ── Model Architecture ────────────────────────────────────
    # Multi-scale: applies convolutions with 3 different kernel sizes simultaneously
    USE_MULTI_SCALE = True
    SCALE_KERNELS = [3, 5, 7]

    # Region-aware: groups 62 EEG channels into 5 anatomical brain regions
    USE_REGION_AWARE = True
    REGION_MAPPING = {
        'frontal':   list(range(1, 15)),   # Channels 1–14: frontal lobe
        'central':   list(range(15, 25)),  # Channels 15–24: central area
        'parietal':  list(range(25, 37)),  # Channels 25–36: parietal lobe
        'temporal':  list(range(37, 49)),  # Channels 37–48: temporal lobe
        'occipital': list(range(49, 63))   # Channels 49–62: occipital lobe
    }

    # ── Training ──────────────────────────────────────────────
    USE_ATTENTION = False    # Self-attention on channels (disabled to reduce compute)
    USE_TEMPORAL = True      # Use multi-scale temporal module
    DROPOUT = 0.3            # Dropout probability to prevent overfitting
    HIDDEN_DIM = 64          # Dimension of hidden representations
    TOPK_GRAPH = 8           # Number of neighbors in sparse graph (top-k sparsification)

    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.0001    # L2 regularization
    EPOCHS = 150
    PATIENCE = 25            # Stop training if no F1 improvement for 25 consecutive epochs
    SEED = 42

    PRINT_FREQUENCY = 10     # Print training stats every N epochs
    SAVE_BEST_ONLY = True    # Only save checkpoint when validation F1 improves


# ─── CREATE OUTPUT DIRECTORIES ───────────────────────────────
for path in [Config.OUTPUT_PATH, Config.MODEL_PATH, Config.RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

# Auto-detect GPU; fall back to CPU if unavailable
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(Config.SEED)

print("=" * 60)
print("BALANCE-OPTIMIZED CONFIGURATION")
print("=" * 60)
print(f"Device: {device}")
print(f"Class weights: {Config.CLASS_WEIGHTS}")
print(f"Batch size: {Config.BATCH_SIZE} ({Config.SAMPLES_PER_CLASS} per class)")
print(f"Focal loss: gamma={Config.FOCAL_GAMMA}, alpha={Config.FOCAL_ALPHA}")
print(f"Label smoothing: {Config.LABEL_SMOOTHING}")
print("=" * 60)


# ─── CELL 2: DATA PREPROCESSOR ──────────────────────────────

class SEEDIVDataPreprocessor:
    """
    Handles loading and preprocessing of SEED-IV dataset files.
    SEED-IV stores EEG features in .mat files (MATLAB format).
    Each file corresponds to one subject-session combination.
    Inside each file, keys like 'de_movingAve1', 'de_movingAve2', ... store
    the Differential Entropy (DE) features for each trial.
    Shape of each trial: (62 channels, T time_steps, 5 frequency_bands)
    """

    def __init__(self, config):
        self.config = config
        self.trial_order = config.TRIAL_EMOTION_MAPPING
        self.emotion_names = config.EMOTION_NAMES

    def load_data(self):
        """
        Scans DATA_PATH for all .mat files, extracts DE features,
        and assembles arrays for data, labels, and subject IDs.

        Returns:
            all_data   : list of np.arrays, each shape (62, T, 5)
            all_labels : np.array of int emotion labels [0–3]
            all_subjects: np.array of int subject IDs [1–15]
        """
        # Collect all .mat file paths (handles both flat and nested directory layouts)
        files = sorted(glob.glob(os.path.join(self.config.DATA_PATH, '*.mat')))
        files.extend(sorted(glob.glob(os.path.join(self.config.DATA_PATH, '*/*.mat'))))
        files = sorted(list(set(files)))

        all_data, all_labels, all_subjects = [], [], []
        time_lengths = []

        print(f"\nLoading {len(files)} files...")

        for file_path in files:
            filename = os.path.basename(file_path)
            try:
                # Subject ID is the number before the first underscore in filename
                subject_id = int(filename.split('_')[0])
                if subject_id > self.config.NUM_SUBJECTS:
                    continue

                data = sio.loadmat(file_path)

                # Extract keys that correspond to DE moving-average features
                de_keys = [k for k in data.keys() if 'de_movingAve' in k]
                de_keys = sorted(de_keys, key=lambda x: int(''.join(filter(str.isdigit, x)) or 0))

                # Process up to 24 trials per file
                for trial_idx, key in enumerate(de_keys[:24]):
                    trial_data = data[key]

                    # ── Shape normalization ─────────────────────────
                    # MATLAB may store data in different axis orders.
                    # Target shape: (62_channels, T_timesteps, 5_bands)
                    if trial_data.ndim == 3:
                        if trial_data.shape[0] == 62 and trial_data.shape[1] == 5 and trial_data.shape[2] != 5:
                            trial_data = trial_data.transpose(0, 2, 1)   # (62, 5, T) → (62, T, 5)
                        elif trial_data.shape[0] == 5 and trial_data.shape[1] == 62:
                            trial_data = trial_data.transpose(1, 2, 0)   # (5, 62, T) → (62, T, 5)
                        elif trial_data.shape[2] == 5 and trial_data.shape[1] == 62:
                            trial_data = trial_data.transpose(1, 0, 2)   # (T, 62, 5) → (62, T, 5)

                    # Skip malformed trials
                    if trial_data.shape[0] != 62 or trial_data.shape[2] != 5:
                        continue

                    time_lengths.append(trial_data.shape[1])
                    all_data.append(trial_data)
                    all_labels.append(self.trial_order[trial_idx])
                    all_subjects.append(subject_id)

            except Exception as e:
                print(f"  Error loading {filename}: {e}")
                continue

        all_labels = np.array(all_labels)
        all_subjects = np.array(all_subjects)

        # Check class balance (ratio of 1.0 = perfectly balanced)
        unique, counts = np.unique(all_labels, return_counts=True)
        balance_ratio = max(counts) / min(counts)

        print(f"\n✓ Loaded {len(all_data)} trials")
        print(f"  Time range: {min(time_lengths)}-{max(time_lengths)} points")
        print(f"  Labels: {dict(zip(unique, counts))}")
        print(f"  Balance ratio (max/min): {balance_ratio:.2f}")

        return all_data, all_labels, all_subjects

    def standardize_time(self, data_list):
        """
        Crops or zero-pads each trial along the time axis to TARGET_TIME_LENGTH.
        - If trial is longer: center-crop to target length
        - If trial is shorter: zero-pad symmetrically on both sides

        This ensures all samples have identical shape for batch processing.

        Returns:
            result: np.array of shape (N, 62, TARGET_TIME_LENGTH, 5)
        """
        target = self.config.TARGET_TIME_LENGTH
        standardized = []

        for trial in data_list:
            current = trial.shape[1]

            if current >= target:
                # Center crop
                start = (current - target) // 2
                end = start + target
                standardized_trial = trial[:, start:end, :]
            else:
                # Symmetric zero-padding
                pad_before = (target - current) // 2
                pad_after = target - current - pad_before
                standardized_trial = np.pad(
                    trial,
                    ((0, 0), (pad_before, pad_after), (0, 0)),
                    mode='constant',
                    constant_values=0
                )

            standardized.append(standardized_trial)

        result = np.stack(standardized, axis=0)
        print(f"\n Time standardized to {target}: {result.shape}")
        return result

    def normalize_per_subject(self, data, subjects):
        """
        Z-score normalizes each subject's data independently using StandardScaler.
        This removes inter-subject signal amplitude differences (e.g. due to electrode
        placement, impedance, or individual EEG amplitude variation).

        Normalization is done across all features (last dimension) independently.

        Returns:
            result: np.array same shape as data, dtype float32
        """
        result = np.zeros_like(data, dtype=np.float32)
        unique_subjects = np.unique(subjects)

        for subject in unique_subjects:
            mask = subjects == subject
            subject_data = data[mask]
            original_shape = subject_data.shape
            flat = subject_data.reshape(-1, original_shape[-1])  # Flatten to (N*62*T, 5)

            scaler = StandardScaler()
            normalized = scaler.fit_transform(flat)
            result[mask] = normalized.reshape(original_shape)

        print(f"\n Normalized: mean={result.mean():.4f}, std={result.std():.4f}")
        return result 


# ─── CELL 3: MULTI-FEATURE EXTRACTOR ────────────────────────

class MultiFeatureExtractor:
    """
    Computes 4 complementary EEG feature types from the raw DE data:

    1. DE  (Differential Entropy): Direct DE features from the dataset
    2. PSD (Power Spectral Density): exp(DE) — since log-PSD ≈ DE for Gaussian signals
    3. SE  (Sample Entropy): Measures signal complexity/irregularity over local windows
    4. PE  (Permutation Entropy): Captures ordinal pattern structure in time series

    All 4 features are concatenated along the last axis:
    (N, 62, T, 5_bands) → (N, 62, T, 20_features)
    """

    def __init__(self, config):
        self.config = config
        self.feature_types = config.FEATURE_TYPES

    def compute_sample_entropy(self, data):
        """
        Approximation of Sample Entropy using local vs global variance ratio.
        SE measures the conditional probability that sequences similar for m points
        remain similar for m+1 points (lower SE = more regular signal).

        Uses a sliding window approach for efficiency.
        """
        samples, channels, time, bands = data.shape
        result = np.zeros_like(data, dtype=np.float32)
        window = min(10, time)  # Local window size for comparison

        for s in range(samples):
            for c in range(channels):
                for b in range(bands):
                    ts = data[s, c, :, b]
                    for t in range(time):
                        start = max(0, t - window)
                        segment = ts[start:t + 1]
                        if len(segment) >= 2:
                            local_var = np.var(segment) + 1e-8
                            global_var = np.var(ts) + 1e-8
                            # Log ratio: high local variance relative to global → high entropy
                            result[s, c, t, b] = -np.log(local_var / global_var + 1e-8)
        return result

    def compute_permutation_entropy(self, data, order=3):
        """
        Permutation Entropy: encodes the ordinal rank pattern of short subsequences.
        For a window of `order` consecutive values, the pattern of their ranks
        captures temporal ordering structure without sensitivity to amplitude.

        Example: [3.1, 1.2, 2.8] → ranks [2, 0, 1] (sorted order indices)
        """
        samples, channels, time, bands = data.shape
        result = np.zeros_like(data, dtype=np.float32)
        perms = list(set(permutations(range(order))))
        n_perms = len(perms)  # order! = 6 for order=3

        for s in range(samples):
            for c in range(channels):
                for b in range(bands):
                    ts = data[s, c, :, b]
                    for t in range(time - order + 1):
                        pattern = tuple(ts[t:t + order])
                        ranked = tuple(np.argsort(np.argsort(pattern)))
                        if ranked in perms:
                            result[s, c, t:t + order, b] += 0.1

        # Normalize by log of number of permutations (max entropy)
        return result / (np.log(n_perms) + 1e-8)

    def extract_features(self, de_data):
        """
        Concatenates all enabled feature types along the last (feature) dimension.

        Input:  de_data shape (N, 62, T, 5)
        Output: concatenated shape (N, 62, T, 20)  [for 4 types × 5 bands]
        """
        all_features = []

        if 'DE' in self.feature_types:
            all_features.append(de_data)  # Original DE features

        if 'PSD' in self.feature_types:
            # Approximate PSD from DE: for Gaussian signals, DE = 0.5 * log(2πe * σ²)
            # So PSD ≈ exp(DE). Clipping prevents numerical overflow.
            psd = np.exp(np.clip(de_data, -5, 5))
            all_features.append(psd)

        if 'SE' in self.feature_types:
            se = self.compute_sample_entropy(de_data)
            se = (se - se.mean()) / (se.std() + 1e-8)  # Z-score normalize
            all_features.append(se)

        if 'PE' in self.feature_types:
            pe = self.compute_permutation_entropy(de_data)
            pe = (pe - pe.mean()) / (pe.std() + 1e-8)
            all_features.append(pe)

        result = np.concatenate(all_features, axis=-1)
        print(f"\n✓ Multi-features extracted: {result.shape}")
        return result


# ─── CELL 4: BALANCED DATASET AND SAMPLERS ──────────────────

class EEGEmotionDataset(Dataset):
    """
    Standard PyTorch Dataset wrapper for EEG data.

    Converts numpy arrays to float32 tensors (data) and int64 tensors (labels).
    Compatible with both DataLoader with batch_sampler and standard DataLoader.
    """

    def __init__(self, data, labels, name=""):
        self.data = torch.FloatTensor(data)
        self.labels = torch.LongTensor(labels)
        print(f"\n{name} Dataset: {len(self.data)} samples")
        print(f"  Shape: {self.data.shape}")
        print(f"  Class distribution: {np.bincount(labels)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


class BalancedBatchSampler:
    """
    Custom batch sampler that ensures each batch contains exactly
    SAMPLES_PER_CLASS examples from every class.

    Why: Standard random sampling produces batches that mirror overall class
    distribution. With 4 balanced classes, this works, but in LOSO splits
    (where one subject is held out), per-class counts in train may differ.
    This sampler explicitly enforces balance every batch.

    How:
    1. Pre-sort dataset indices by class
    2. Each batch: pick SAMPLES_PER_CLASS indices from each class, shuffle them
    3. Number of batches = min_class_count // SAMPLES_PER_CLASS
    """

    def __init__(self, labels, batch_size, num_classes):
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.samples_per_class = batch_size // num_classes  # e.g. 128//4 = 32

        self.class_indices = []
        self.class_counts = []

        for c in range(num_classes):
            idx = np.where(self.labels == c)[0]
            np.random.shuffle(idx)
            self.class_indices.append(idx)
            self.class_counts.append(len(idx))

        # Use the smallest class to determine epoch length (no oversampling)
        min_class_samples = min(self.class_counts)
        self.num_batches = min_class_samples // self.samples_per_class

        print(f"\n📊 Balanced sampler created:")
        print(f"  Samples per class per batch: {self.samples_per_class}")
        print(f"  Batches per epoch: {self.num_batches}")
        print(f"  Total samples per epoch: {self.num_batches * batch_size}")

    def __iter__(self):
        """Yield one list of indices per batch, balanced across classes."""
        class_pos = [0] * self.num_classes
        for c in range(self.num_classes):
            np.random.shuffle(self.class_indices[c])  # Re-shuffle at start of each epoch

        for _ in range(self.num_batches):
            batch = []

            for c in range(self.num_classes):
                indices = self.class_indices[c]
                start = class_pos[c]

                # Wrap around if we've used all samples for this class
                if start + self.samples_per_class > len(indices):
                    np.random.shuffle(indices)
                    start = 0
                    class_pos[c] = self.samples_per_class
                else:
                    class_pos[c] += self.samples_per_class

                batch.extend(indices[start:start + self.samples_per_class])

            np.random.shuffle(batch)  # Shuffle so classes aren't in blocks
            yield batch

    def __len__(self):
        return self.num_batches


class WeightedSampler:
    """
    Fallback sampler using PyTorch's WeightedRandomSampler.
    Assigns each sample a probability inversely proportional to its class size.
    Rarer classes are oversampled to compensate for imbalance.
    """

    def __init__(self, labels, num_classes, config):
        self.labels = np.array(labels)
        self.num_classes = num_classes

        # Inverse frequency weighting
        class_counts = np.bincount(labels, minlength=num_classes)
        class_weights = 1.0 / (class_counts + 1e-8)
        class_weights = class_weights / class_weights.sum()

        sample_weights = class_weights[labels]  # Assign each sample its class weight

        self.sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(labels) * 2,  # Oversample to 2× dataset size
            replacement=True
        )

        print(f"\n📊 Weighted sampler created:")
        print(f"  Class weights: {class_weights}")

    def get_sampler(self):
        return self.sampler


# ─── CELL 5: LOSS FUNCTIONS ──────────────────────────────────

class FocalLoss(nn.Module):
    """
    Focal Loss: addresses class imbalance by down-weighting easy examples.

    Standard Cross-Entropy: CE(p_t) = -log(p_t)
    Focal Loss: FL(p_t) = -α_t × (1 - p_t)^γ × log(p_t)

    - (1 - p_t)^γ: when model is confident (p_t → 1), this term → 0, reducing loss
                   when model is uncertain (p_t → 0), this term → 1, preserving full loss
    - α_t: per-class weight to additionally handle imbalance
    - label_smoothing: replaces hard 0/1 targets with soft (ε/K-1) / (1-ε) targets

    Focal Loss is especially useful when some classes are consistently easier
    than others (e.g. Neutral may be easy to recognize from flat EEG signals).
    """

    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        if self.label_smoothing > 0:
            # Create soft label distribution
            num_classes = inputs.size(-1)
            smooth_targets = torch.zeros_like(inputs)
            smooth_targets.fill_(self.label_smoothing / (num_classes - 1))   # Small prob for wrong classes
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.label_smoothing)  # High prob for correct

            log_probs = F.log_softmax(inputs, dim=-1)
            ce_loss = -(smooth_targets * log_probs).sum(dim=-1)  # Per-sample cross-entropy
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        pt = torch.exp(-ce_loss)  # Probability of correct class prediction

        if self.alpha is not None:
            if isinstance(self.alpha, (list, tuple)):
                alpha = torch.tensor(self.alpha, dtype=torch.float32).to(inputs.device)
                alpha_t = alpha[targets]   # Gather per-sample alpha values
            else:
                alpha_t = self.alpha[targets]
            focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        else:
            focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss  # Return per-sample losses


class ContrastiveLoss(nn.Module):
    """
    NT-Xent (Normalized Temperature-scaled Cross Entropy) Contrastive Loss.

    Goal: Pull embeddings of same-class samples together; push different-class apart.
    This improves the separability of the learned feature space and complements
    the classification loss.

    How:
    - For each sample, positives = other samples from same class
    - Negatives = all other samples
    - Loss maximizes similarity to positives relative to all negatives
    - temperature: lower = sharper distinctions, higher = more diffuse

    Used as an auxiliary loss: total_loss = cls_loss + 0.2 × contrastive_loss
    """

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels=None):
        features = F.normalize(features, dim=1)  # L2-normalize embeddings to unit sphere

        # Cosine similarity matrix (scaled by temperature)
        similarity = torch.matmul(features, features.T) / self.temperature

        batch_size = features.size(0)

        if labels is not None:
            # Create binary mask: 1 where same class, 0 otherwise
            labels = labels.view(-1, 1)
            mask = torch.eq(labels, labels.T).float()
            mask = mask - torch.eye(batch_size).to(mask.device)  # Exclude self-similarity

            pos_weight = mask.sum(dim=1)  # Number of positive pairs per sample

            # Average positive similarity vs all negative similarities
            pos_similarity = (similarity * mask).sum(dim=1) / (pos_weight + 1e-8)
            neg_similarity = similarity * (1 - mask - torch.eye(batch_size).to(mask.device))

            loss = -torch.log(
                torch.exp(pos_similarity) /
                (torch.exp(pos_similarity) + neg_similarity.exp().sum(dim=1))
            ).mean()
        else:
            # Fallback: treat diagonal pairs as positives (self-supervised style)
            labels = torch.arange(batch_size).to(features.device)
            loss = F.cross_entropy(similarity, labels)

        return loss


# ─── CELL 6: DATA AUGMENTATION ───────────────────────────────

class BalancedAugmentation:
    """
    Applies one of 4 stochastic augmentations during training.
    All augmentations preserve label consistency or produce soft (mixed) labels.

    1. Gaussian Noise:   Adds small random perturbations to raw EEG features
    2. Mixup:            Linearly interpolates two samples; labels become weighted averages
    3. CutMix:           Replaces a temporal segment with that of another sample
    4. Time Warp:        Stretches or compresses the time axis by a random factor

    Augmentation is applied with 70% probability per batch during training.
    During validation/testing, augmentation is disabled via eval() mode.
    """

    def __init__(self, config):
        self.config = config
        self.noise_std = config.NOISE_STD
        self.mixup_alpha = config.MIXUP_ALPHA
        self.cutmix_prob = config.CUTMIX_PROB
        self.training = True

    def train(self):
        """Enable augmentation (for training phase)."""
        self.training = True

    def eval(self):
        """Disable augmentation (for validation/test phase)."""
        self.training = False

    def gaussian_noise(self, x):
        """Add zero-mean Gaussian noise with std=NOISE_STD to all elements."""
        noise = torch.randn_like(x) * self.noise_std
        return x + noise

    def mixup(self, x, y):
        """
        Mixup: blends pairs of samples and their labels.
        λ ~ Beta(α, α); mixed_x = λ*x_i + (1-λ)*x_j
        Encourages linear interpolation between class decision boundaries.
        """
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)  # Random pairing

        # Sample λ from Beta distribution; ensure dominant sample has weight ≥ 0.5
        lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
        lam = max(lam, 1 - lam)

        mixed_x = lam * x + (1 - lam) * x[index]

        # Convert hard labels to one-hot for mixing
        if y.dim() == 1:
            y_onehot = F.one_hot(y, num_classes=self.config.NUM_CLASSES).float()
            y_mixed = lam * y_onehot + (1 - lam) * y_onehot[index]
        else:
            y_mixed = lam * y + (1 - lam) * y[index]

        return mixed_x, y_mixed

    def cutmix(self, x, y):
        """
        CutMix: replaces a temporal segment of one sample with that of another.
        Encourages the model to classify based on partial temporal information,
        improving temporal robustness.
        """
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)

        # Pick a random cut point along the time dimension
        time_dim = x.size(3)   # x shape: (B, Channels, Features, Time)
        cut_ratio = np.random.beta(1, 1)
        cut_point = int(time_dim * cut_ratio)

        mixed_x = x.clone()
        mixed_x[:, :, :, cut_point:] = x[index, :, :, cut_point:]  # Replace tail segment

        # Label is proportional to how much of each sample remains
        lam = cut_point / time_dim
        if y.dim() == 1:
            y_onehot = F.one_hot(y, num_classes=self.config.NUM_CLASSES).float()
            y_mixed = lam * y_onehot + (1 - lam) * y_onehot[index]
        else:
            y_mixed = lam * y + (1 - lam) * y[index]

        return mixed_x, y_mixed

    def time_warp(self, x, y):
        """
        Time Warp: randomly stretches or compresses the temporal axis.
        warp_factor ∈ [0.8, 1.2] → 20% temporal distortion max.
        After warping, the signal is resampled back to original length.
        """
        batch_size, channels, features, time = x.shape

        warp_factor = 1.0 + np.random.uniform(-0.2, 0.2)
        new_time = int(time * warp_factor)

        # Interpolate along time dimension
        x_reshaped = x.view(batch_size * channels, features, time)
        x_warped = F.interpolate(x_reshaped, size=new_time, mode='linear', align_corners=False)

        if new_time > time:
            # Crop center if stretched
            start = (new_time - time) // 2
            x_warped = x_warped[:, :, start:start + time]
        elif new_time < time:
            # Zero-pad if compressed
            pad = time - new_time
            x_warped = F.pad(x_warped, (pad // 2, pad - pad // 2))

        return x_warped.view(batch_size, channels, features, time), y

    def __call__(self, x, y):
        """
        With 70% probability during training, randomly pick and apply one augmentation.
        Returns (x, y) unchanged during eval or when augmentation is skipped.
        """
        if not self.training:
            return x, y

        if np.random.random() > 0.3:  # 70% chance to augment
            aug_type = np.random.choice(self.config.AUGMENTATION_TYPES)

            if aug_type == 'noise':
                return self.gaussian_noise(x), y
            elif aug_type == 'mixup':
                return self.mixup(x, y)
            elif aug_type == 'cutmix':
                return self.cutmix(x, y)
            elif aug_type == 'time_warp':
                return self.time_warp(x, y)

        return x, y


# ─── CELL 7: GRAPH NEURAL NETWORK COMPONENTS ─────────────────

class GraphConvolution(nn.Module):
    """
    Standard Graph Convolution Layer (Kipf & Welling, 2017):
        H' = A · H · W + b

    where:
        A = adjacency matrix (learned from data)
        H = node feature matrix (EEG channels)
        W = learnable weight matrix
        b = bias vector

    Captures spatial relationships between EEG electrodes (channels).
    """

    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        nn.init.xavier_uniform_(self.weight)  # Xavier init for stable gradients
        nn.init.zeros_(self.bias)

    def forward(self, x, adj):
        # x: (batch, nodes, in_features), adj: (batch, nodes, nodes)
        return torch.matmul(adj, torch.matmul(x, self.weight)) + self.bias


class SelfOrganizedGraphConstruction(nn.Module):
    """
    Dynamically constructs a sparse graph from node features.

    SOGNN approach:
    1. Project each node to a bottleneck space via a linear layer + tanh
    2. Compute pairwise similarity scores (dot product → softmax → adjacency matrix)
    3. Sparsify: keep only top-k connections per node (top-k sparsification)
    4. Apply graph convolution using the constructed adjacency

    This lets the model learn which EEG channels influence each other,
    rather than using a fixed anatomical connectivity graph.

    Parameters:
        in_features  : dimension of input node features
        bn_features  : bottleneck dimension for graph construction
        out_features : output feature dimension after graph conv
        topk         : number of neighbors per node to keep
    """

    def __init__(self, in_features, bn_features, out_features, topk=10):
        super().__init__()
        self.topk = topk
        self.bottleneck = nn.Linear(in_features, bn_features)  # For similarity projection
        self.graph_conv = GraphConvolution(in_features, out_features)

    def forward(self, x):
        # x: (batch, 62_channels, features)

        # 1. Project to bottleneck for similarity computation
        xa = torch.tanh(self.bottleneck(x))

        # 2. Compute soft adjacency: A = softmax(xa · xa^T)
        adj = F.softmax(torch.matmul(xa, xa.transpose(1, 2)), dim=2)

        # 3. Top-k sparsification: zero out all but top-k edges per node
        mask = torch.zeros_like(adj)
        _, idx = adj.topk(self.topk, dim=2)
        mask.scatter_(2, idx, 1.0)

        # 4. Graph convolution with sparsified adjacency
        return F.relu(self.graph_conv(x, adj * mask))


# ─── CELL 8: REGION-AWARE PROCESSOR ──────────────────────────

class RegionAwareProcessor(nn.Module):
    """
    Groups EEG channels into 5 anatomical brain regions and processes
    each region's channels jointly with a separate linear projection.

    Brain regions and their functions:
    - Frontal:   Decision making, working memory, emotional regulation
    - Central:   Motor control, somatosensory processing
    - Parietal:  Spatial awareness, attention, integration of sensory info
    - Temporal:  Auditory processing, language, memory consolidation
    - Occipital: Visual processing

    After per-region projection, all regions are fused into a single vector
    and added (as a residual) to the channel-averaged global representation.

    This architecture encodes domain knowledge about EEG spatial structure.
    """

    def __init__(self, config, input_dim):
        super().__init__()
        self.config = config
        self.regions = config.REGION_MAPPING

        self.region_projections = nn.ModuleDict()

        for region_name, channels in self.regions.items():
            # Convert 1-indexed channel IDs to 0-indexed
            channels = [c - 1 for c in channels if c - 1 < config.NUM_CHANNELS]
            if channels:
                # Each region: concatenated channel features → single region vector
                self.region_projections[region_name] = nn.Linear(
                    len(channels) * input_dim, input_dim
                )

        # Fuse all region representations into one vector
        self.fusion = nn.Linear(len(self.regions) * input_dim, input_dim)
        self.norm = nn.LayerNorm(input_dim)

    def forward(self, x):
        # x: (batch, 62_channels, features)
        batch_size = x.size(0)
        region_outputs = []

        for region_name, channels in self.regions.items():
            channels = [c - 1 for c in channels if c - 1 < x.size(1)]
            if not channels:
                continue

            region_data = x[:, channels, :]                      # Select region channels
            region_flat = region_data.reshape(batch_size, -1)     # Flatten: (B, n_ch * features)
            region_proj = self.region_projections[region_name](region_flat)
            region_outputs.append(region_proj)

        if region_outputs:
            combined = torch.cat(region_outputs, dim=1)   # (B, 5 * features)
            fused = self.fusion(combined)                  # (B, features)
            return self.norm(fused + x.mean(dim=1))        # Residual connection
        else:
            return x.mean(dim=1)


# ─── CELL 9: MULTI-SCALE TEMPORAL MODULE ─────────────────────

class MultiScaleTemporal(nn.Module):
    """
    Captures temporal EEG dynamics at multiple time scales simultaneously
    using parallel 1D convolutions with different kernel sizes.

    Kernel sizes = [3, 5, 7] capture:
    - 3: short-range, rapid EEG fluctuations
    - 5: mid-range temporal patterns
    - 7: longer-range, slower dynamics (e.g. alpha oscillations)

    All scale outputs are concatenated and fused with a linear layer.
    This is applied independently per EEG channel, then averaged across channels.

    Input:  (batch, 62_channels, 16_features, T_timesteps)
    Output: (batch, 62_channels, hidden_dim)
    """

    def __init__(self, input_dim, hidden_dim, kernel_sizes=[3, 5, 7]):
        super().__init__()
        self.kernel_sizes = kernel_sizes
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        num_scales = len(kernel_sizes)
        self.out_channels_per_scale = hidden_dim // num_scales   # e.g. 64//3 = 21
        self.conv_out_dim = self.out_channels_per_scale * num_scales

        # One 1D convolution per scale; padding='k//2' preserves time dimension
        self.convs = nn.ModuleList([
            nn.Conv1d(input_dim, self.out_channels_per_scale,
                      kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])

        self.fusion = nn.Linear(self.conv_out_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x: (batch, channels, input_dim, time)
        batch, channels, input_dim, time = x.shape
        x_reshaped = x.reshape(batch * channels, input_dim, time)   # Merge batch+channels for conv

        multi_scale_outputs = []
        for conv in self.convs:
            out = F.relu(conv(x_reshaped))   # (B*C, out_ch_per_scale, time)
            multi_scale_outputs.append(out)

        combined = torch.cat(multi_scale_outputs, dim=1)  # (B*C, conv_out_dim, time)
        combined = combined.permute(0, 2, 1)               # (B*C, time, conv_out_dim)

        attended = self.fusion(combined)    # (B*C, time, hidden_dim)
        attended = self.norm(attended)

        output = attended.mean(dim=1)                       # Average over time → (B*C, hidden_dim)
        output = output.view(batch, channels, -1)           # Reshape to (B, C, hidden_dim)

        return output


# ─── CELL 10: BALANCED SOGNN MODEL ───────────────────────────

class BalancedSOGNN(nn.Module):
    """
    Self-Organized Graph Neural Network (SOGNN) with class balance optimization.

    Architecture (3 hierarchical blocks):
    ┌─────────────────────────────────────────────────────────┐
    │ Input: (B, 62, T=35, 20_features)                       │
    │                                                          │
    │ Block 1: Conv2D(1→16) + BN + Pool → (B*62, 16, t1, 1)  │
    │   ├── RegionAwareProcessor (spatial grouping)            │
    │   ├── SOGNN Graph Convolution                           │
    │   └── MultiScaleTemporal (temporal extraction)          │
    │                                                          │
    │ Block 2: Conv2D(16→32) + BN + Pool → (B*62, 32, t2, 1) │
    │   └── [same as Block 1]                                 │
    │                                                          │
    │ Block 3: Conv2D(32→64) + BN → (B*62, 64, t3, 1)        │
    │   └── [same as Block 1]                                 │
    │                                                          │
    │ Concatenate temporal outputs: (B, 62, 192)              │
    │ Mean pool over channels: (B, 192)                        │
    │ Dropout + Linear → (B, 4) logits                        │
    └─────────────────────────────────────────────────────────┘

    The classifier bias is initialized based on class weights to give
    balanced initial predictions before any training occurs.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.channels = config.NUM_CHANNELS      # 62
        self.num_features = config.TOTAL_FEATURES  # 20
        self.num_classes = config.NUM_CLASSES    # 4

        print("\n" + "=" * 50)
        print("BUILDING BALANCED SOGNN")
        print("=" * 50)

        # ── Compute intermediate time dimensions after pooling ──
        # Conv2D with kernel 5, no padding → time reduces by 4 each time
        # MaxPool(1, 2) halves the time dimension
        t1 = (35 - 5 + 1) // 2    # After conv1 + pool1: (31)//2 = 15
        t2 = (t1 - 5 + 1) // 2   # After conv2 + pool2: (11)//2 = 5
        t3 = (t2 - 5 + 1)         # After conv3 (no pool): 5-4 = 1
        self.t1, self.t2, self.t3 = t1, t2, t3

        print(f"Time progression: 35 → {t1} → {t2} → {t3}")

        # ── CNN Blocks ────────────────────────────────────────
        # Conv2D processes (feature_dim, time) for all channels in parallel
        # kernel (num_features, 5): collapses all features; 5-point temporal convolution
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(self.num_features, 5))
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d((1, 2))

        self.conv2 = nn.Conv2d(16, 32, kernel_size=(1, 5))
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d((1, 2))

        self.conv3 = nn.Conv2d(32, 64, kernel_size=(1, 5))
        self.bn3 = nn.BatchNorm2d(64)

        # Flattened feature dimensions after each block (used for graph construction)
        self.feat1 = 16 * t1   # 16 channels × t1 time steps
        self.feat2 = 32 * t2
        self.feat3 = 64 * t3

        print(f"Feature dimensions: {self.feat1} → {self.feat2} → {self.feat3}")

        # ── Region-Aware Processors ───────────────────────────
        if config.USE_REGION_AWARE:
            self.region_processor1 = RegionAwareProcessor(config, self.feat1)
            self.region_processor2 = RegionAwareProcessor(config, self.feat2)
            self.region_processor3 = RegionAwareProcessor(config, self.feat3)

        # ── Graph Modules (one per block) ─────────────────────
        self.graph1 = SelfOrganizedGraphConstruction(self.feat1, 32, 32, config.TOPK_GRAPH)
        self.graph2 = SelfOrganizedGraphConstruction(self.feat2, 32, 32, config.TOPK_GRAPH)
        self.graph3 = SelfOrganizedGraphConstruction(self.feat3, 32, 32, config.TOPK_GRAPH)

        # ── Temporal Modules (one per block) ──────────────────
        if config.USE_TEMPORAL and config.USE_MULTI_SCALE:
            self.temporal1 = MultiScaleTemporal(16, config.HIDDEN_DIM, config.SCALE_KERNELS)
            self.temporal2 = MultiScaleTemporal(16, config.HIDDEN_DIM, config.SCALE_KERNELS)
            self.temporal3 = MultiScaleTemporal(16, config.HIDDEN_DIM, config.SCALE_KERNELS)

        # ── Channel reducers for blocks 2 & 3 ─────────────────
        # temporal modules expect 16 input channels, so we reduce 32→16 and 64→16
        self.reduce_channels2 = nn.Conv2d(32, 16, kernel_size=1)
        self.reduce_channels3 = nn.Conv2d(64, 16, kernel_size=1)

        # ── Classifier ────────────────────────────────────────
        feature_dim = config.HIDDEN_DIM * 3  # 64 × 3 = 192 (concat from 3 blocks)
        self.dropout = nn.Dropout(config.DROPOUT)
        self.classifier = nn.Linear(feature_dim, self.num_classes)

        # Initialize bias with log of normalized class weights
        # This biases initial predictions proportionally to expected class frequencies
        with torch.no_grad():
            self.classifier.bias.data = torch.log(
                torch.tensor(config.CLASS_WEIGHTS) / sum(config.CLASS_WEIGHTS)
            )

        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\nTotal trainable parameters: {total_params:,}")
        print("=" * 50 + "\n")

    def forward(self, x, return_features=False):
        """
        Forward pass.

        Args:
            x             : input tensor (B, 62, T, 20)
            return_features: if True, also return the pre-classifier embedding

        Returns:
            output          : class logits (B, 4)
            features (opt.) : embedding (B, 192)
        """
        batch_size = x.size(0)

        # ── Reshape for Conv2D ────────────────────────────────
        # Input: (B, 62, T, 20) → permute → (B, 62, 20, T) → reshape → (B*62, 1, 20, T)
        x = x.permute(0, 1, 3, 2)
        x = x.reshape(-1, 1, self.num_features, x.size(3))

        # ── Block 1 ───────────────────────────────────────────
        x = F.relu(self.bn1(self.conv1(x)))   # (B*62, 16, 1, t1)
        x = self.pool1(x)
        x_temp1 = x.clone()                   # Save for temporal processing

        x1 = x.view(batch_size, self.channels, -1)  # (B, 62, 16*t1)
        if self.config.USE_REGION_AWARE:
            x1 = x1 + self.region_processor1(x1).unsqueeze(1)  # Add region-aware residual
        x1_graph = self.graph1(x1)   # Graph conv: (B, 62, 32)

        # ── Block 2 ───────────────────────────────────────────
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x_temp2 = x.clone()

        x2 = x.view(batch_size, self.channels, -1)
        if self.config.USE_REGION_AWARE:
            x2 = x2 + self.region_processor2(x2).unsqueeze(1)
        x2_graph = self.graph2(x2)

        # ── Block 3 ───────────────────────────────────────────
        x = F.relu(self.bn3(self.conv3(x)))
        x_temp3 = x.clone()

        x3 = x.view(batch_size, self.channels, -1)
        if self.config.USE_REGION_AWARE:
            x3 = x3 + self.region_processor3(x3).unsqueeze(1)
        x3_graph = self.graph3(x3)

        # ── Temporal Processing ───────────────────────────────
        if self.config.USE_TEMPORAL:
            x_temp1 = x_temp1.view(batch_size, self.channels, 16, self.t1)
            x1_out = self.temporal1(x_temp1)   # (B, 62, 64)

            x_temp2 = self.reduce_channels2(x_temp2)
            x_temp2 = x_temp2.view(batch_size, self.channels, 16, self.t2)
            x2_out = self.temporal2(x_temp2)

            x_temp3 = self.reduce_channels3(x_temp3)
            x_temp3 = x_temp3.view(batch_size, self.channels, 16, self.t3)
            x3_out = self.temporal3(x_temp3)

            # Concat temporal outputs from all 3 blocks → (B, 62, 192)
            x_combined = torch.cat([x1_out, x2_out, x3_out], dim=2)
            x_combined = x_combined.mean(dim=1)   # Average over 62 channels → (B, 192)
        else:
            # Fallback: use graph outputs
            x_combined = torch.cat([x1_graph, x2_graph, x3_graph], dim=2)
            x_combined = x_combined.mean(dim=1)

        x_combined = self.dropout(x_combined)
        features = x_combined
        output = self.classifier(features)   # (B, 4)

        if return_features:
            return output, features
        return output


# ─── CELL 11: BALANCED TRAINER ───────────────────────────────

class BalancedTrainer:
    """
    Manages the full training loop for one LOSO fold.

    Key components:
    - BalancedBatchSampler: guarantees equal class representation per batch
    - FocalLoss: downweights easy examples to focus on hard ones
    - ContrastiveLoss: auxiliary loss to improve feature space separability
    - CosineAnnealingWarmRestarts: LR scheduler that periodically resets LR
    - BalancedAugmentation: stochastic EEG augmentation per batch
    - Gradient clipping: prevents exploding gradients (max_norm=5.0)
    - Early stopping: stops training if val F1 doesn't improve for PATIENCE epochs

    Model checkpoints are saved to MODEL_PATH/balanced_model_fold{N}.pth
    """

    def __init__(self, model, train_dataset, val_loader, config, device, fold=None):
        self.model = model
        self.train_dataset = train_dataset
        self.val_loader = val_loader
        self.config = config
        self.device = device
        self.fold = fold

        train_labels = train_dataset.labels.numpy()

        # ── DataLoader with balanced sampling ─────────────────
        if config.USE_BALANCED_SAMPLING:
            self.batch_sampler = BalancedBatchSampler(
                train_labels, config.BATCH_SIZE, config.NUM_CLASSES
            )
            self.train_loader = DataLoader(
                train_dataset, batch_sampler=self.batch_sampler, num_workers=0
            )
        else:
            sampler = WeightedSampler(train_labels, config.NUM_CLASSES, config)
            self.train_loader = DataLoader(
                train_dataset, batch_size=config.BATCH_SIZE,
                sampler=sampler.get_sampler(), num_workers=0
            )

        # ── Loss Function ──────────────────────────────────────
        if config.USE_FOCAL_LOSS:
            self.criterion = FocalLoss(
                gamma=config.FOCAL_GAMMA,
                alpha=config.FOCAL_ALPHA,
                label_smoothing=config.LABEL_SMOOTHING
            )
            print(f"\n📉 Using Focal Loss: gamma={config.FOCAL_GAMMA}, alpha={config.FOCAL_ALPHA}")
        else:
            weights = torch.FloatTensor(config.CLASS_WEIGHTS).to(device)
            self.criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=config.LABEL_SMOOTHING)
            print(f"\n📉 Using Weighted CE: weights={config.CLASS_WEIGHTS}")

        # ── Optimizer: AdamW (Adam with decoupled weight decay) ──
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )

        # ── LR Scheduler: Cosine Annealing with Warm Restarts ──
        # LR follows a cosine curve from LEARNING_RATE → eta_min, then restarts
        # T_0=10: first restart after 10 epochs, T_mult=2: each cycle is 2× longer
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=10, T_mult=2, eta_min=1e-6
        )

        # ── Contrastive Loss ───────────────────────────────────
        if config.USE_CONTRASTIVE:
            self.contrastive = ContrastiveLoss(temperature=config.CONTRASTIVE_TEMPERATURE).to(device)
            self.contrastive_weight = config.CONTRASTIVE_WEIGHT
            print(f"🔄 Using Contrastive Learning: weight={self.contrastive_weight}")

        self.augmentation = BalancedAugmentation(config)

        # ── Tracking ───────────────────────────────────────────
        self.best_val_acc = 0
        self.best_val_f1 = 0
        self.patience = 0
        self.save_path = os.path.join(config.MODEL_PATH, f'balanced_model_fold{fold}.pth')

        self.train_class_acc_history = []
        self.val_class_acc_history = []

        print(f"\n✓ Balanced Trainer ready - Fold {fold}")
        print(f"  Train: {len(train_dataset)} samples")
        print(f"  Val: {len(val_loader.dataset)} samples")

    def train_epoch(self):
        """
        Runs one full training epoch.
        Returns: (avg_loss, per_class_accuracy_list)
        """
        self.model.train()
        self.augmentation.train()
        total_loss = 0
        class_correct = [0] * self.config.NUM_CLASSES
        class_total = [0] * self.config.NUM_CLASSES

        for batch_x, batch_y in self.train_loader:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            # Apply random augmentation (may change batch_y to soft labels)
            if self.config.USE_AUGMENTATION:
                batch_x, batch_y = self.augmentation(batch_x, batch_y)

            self.optimizer.zero_grad()

            # Forward pass; request features for contrastive loss
            output, features = self.model(batch_x, return_features=True)

            # ── Compute Loss ────────────────────────────────────
            if batch_y.dim() > 1:
                # Soft labels from Mixup/CutMix: compute weighted sum of per-class losses
                loss = 0
                for c in range(self.config.NUM_CLASSES):
                    if batch_y[:, c].sum() > 0:
                        class_loss = self.criterion(
                            output, torch.full_like(batch_y[:, c].long(), c)
                        )
                        loss += class_loss * batch_y[:, c].mean()
            else:
                # Hard labels: standard focal loss
                cls_loss = self.criterion(output, batch_y)
                if self.config.USE_CONTRASTIVE and self.contrastive_weight > 0:
                    contrast_loss = self.contrastive(features, batch_y)
                    loss = cls_loss + self.contrastive_weight * contrast_loss
                else:
                    loss = cls_loss

            # ── Backward Pass ─────────────────────────────────
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)  # Prevent gradient explosion
            self.optimizer.step()

            total_loss += loss.item()

            # Track per-class training accuracy
            with torch.no_grad():
                preds = output.argmax(1)
                if batch_y.dim() == 1:
                    for c in range(self.config.NUM_CLASSES):
                        mask = (batch_y == c)
                        class_total[c] += mask.sum().item()
                        class_correct[c] += (preds[mask] == c).sum().item()

        per_class_acc = [
            100 * class_correct[c] / class_total[c] if class_total[c] > 0 else 0
            for c in range(self.config.NUM_CLASSES)
        ]
        self.train_class_acc_history.append(per_class_acc)

        return total_loss / len(self.train_loader), per_class_acc

    def validate(self):
        """
        Evaluates model on the validation set.
        Returns: (avg_loss, accuracy, macro_f1, per_class_acc, predictions, targets)
        """
        self.model.eval()
        self.augmentation.eval()
        total_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch_x, batch_y in self.val_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)

                output = self.model(batch_x)
                loss = self.criterion(output, batch_y)

                total_loss += loss.item()
                pred = output.argmax(1)
                correct += (pred == batch_y).sum().item()
                total += batch_y.size(0)

                all_preds.extend(pred.cpu().numpy())
                all_targets.extend(batch_y.cpu().numpy())

        val_acc = 100 * correct / total
        val_f1 = f1_score(all_targets, all_preds, average='macro')

        cm = confusion_matrix(all_targets, all_preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
        self.val_class_acc_history.append(per_class_acc)

        return total_loss / len(self.val_loader), val_acc, val_f1, per_class_acc, all_preds, all_targets

    def train(self):
        """
        Full training loop with early stopping.

        - Runs train_epoch() and validate() each epoch
        - Saves checkpoint when validation F1 improves
        - Stops training when patience threshold is exceeded
        - Returns the best validation F1 score achieved
        """
        print(f"\n{'=' * 60}")
        print(f"TRAINING FOLD {self.fold} (BALANCED)")
        print(f"{'=' * 60}")

        for epoch in range(self.config.EPOCHS):
            train_loss, per_class_train = self.train_epoch()
            val_loss, val_acc, val_f1, per_class_val, val_preds, val_targets = self.validate()

            self.scheduler.step()
            current_lr = self.optimizer.param_groups[0]['lr']

            # Print per-class accuracies every N epochs
            if epoch % self.config.PRINT_FREQUENCY == 0:
                print(f"\nEpoch {epoch:3d}:")
                print(f"  Train: Neu={per_class_train[0]:.1f}%, Sad={per_class_train[1]:.1f}%, "
                      f"Fear={per_class_train[2]:.1f}%, Happy={per_class_train[3]:.1f}%")
                print(f"  Val:   Neu={per_class_val[0]:.1f}%, Sad={per_class_val[1]:.1f}%, "
                      f"Fear={per_class_val[2]:.1f}%, Happy={per_class_val[3]:.1f}%")
                print(f"  Val Acc={val_acc:.2f}%, Val F1={val_f1:.4f}, LR={current_lr:.2e}")

            # Save checkpoint if this is the best model so far (by val F1)
            if val_f1 > self.best_val_f1:
                self.best_val_f1 = val_f1
                self.best_val_acc = val_acc
                self.patience = 0

                if self.config.SAVE_BEST_ONLY:
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': self.model.state_dict(),
                        'optimizer_state_dict': self.optimizer.state_dict(),
                        'val_acc': val_acc,
                        'val_f1': val_f1,
                        'per_class_val': per_class_val,  # NOTE: may be numpy array → allowlisted above
                    }, self.save_path)

                    print(f"  → New best! (F1={val_f1:.4f})")
            else:
                self.patience += 1
                if self.patience >= self.config.PATIENCE:
                    print(f"\nEarly stopping at epoch {epoch}")
                    break

        print(f"\n✓ Best Validation F1: {self.best_val_f1:.4f} (Acc: {self.best_val_acc:.2f}%)")
        return self.best_val_f1


# ─── CELL 12: LOSO CROSS-VALIDATION ──────────────────────────

def run_balanced_loso(data, labels, subjects, config, device):
    """
    Leave-One-Subject-Out (LOSO) cross-validation.

    For each of the 15 subjects:
    1. Hold out that subject as the test set
    2. Split remaining 14 subjects into train (85%) and val (15%) using stratified split
    3. Train a fresh SOGNN model on train, monitor on val
    4. Evaluate best checkpoint on the held-out test subject
    5. Record accuracy, F1, per-class accuracy, confusion matrix

    LOSO is the gold standard for EEG BCI evaluation because it directly measures
    how well the model generalizes to completely unseen subjects (cross-subject).

    Returns:
        results          : list of dicts with per-fold accuracy/F1
        per_class_results: list of dicts with per-fold per-class accuracy
    """
    unique_subjects = np.unique(subjects)
    results = []
    per_class_results = []
    confusion_matrices = []

    print("\n" + "=" * 60)
    print("BALANCED LOSO CROSS-VALIDATION")
    print("=" * 60)

    for fold, test_subj in enumerate(unique_subjects):
        print(f"\n{'=' * 60}")
        print(f"FOLD {fold + 1}/{len(unique_subjects)}: Test Subject {test_subj}")
        print(f"{'=' * 60}")

        # ── Split data ────────────────────────────────────────
        test_mask = (subjects == test_subj)
        train_val_mask = ~test_mask

        X_test, y_test = data[test_mask], labels[test_mask]
        X_train_val, y_train_val = data[train_val_mask], labels[train_val_mask]

        # ── Stratified validation split (15% of train subjects) ─
        # StratifiedShuffleSplit ensures each class is proportionally represented in val
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=config.SEED + fold)
        train_idx, val_idx = next(sss.split(X_train_val, y_train_val))
        X_train, y_train = X_train_val[train_idx], y_train_val[train_idx]
        X_val, y_val = X_train_val[val_idx], y_train_val[val_idx]

        # Print class distribution for this fold
        train_dist = np.bincount(y_train, minlength=config.NUM_CLASSES)
        val_dist = np.bincount(y_val, minlength=config.NUM_CLASSES)
        test_dist = np.bincount(y_test, minlength=config.NUM_CLASSES)

        print(f"\nClass distribution (Neu/Sad/Fear/Happy):")
        print(f"  Train: {train_dist}")
        print(f"  Val:   {val_dist}")
        print(f"  Test:  {test_dist}")

        balance_ratio = max(train_dist) / min(train_dist)
        print(f"  Train balance ratio: {balance_ratio:.2f}")

        # ── Create PyTorch Datasets ───────────────────────────
        train_dataset = EEGEmotionDataset(X_train, y_train, "Train")
        val_dataset = EEGEmotionDataset(X_val, y_val, "Val")
        test_dataset = EEGEmotionDataset(X_test, y_test, "Test")

        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE)

        # ── Build and Train Model ─────────────────────────────
        set_seed(config.SEED + fold)  # Different seed per fold for diversity
        model = BalancedSOGNN(config).to(device)
        trainer = BalancedTrainer(model, train_dataset, val_loader, config, device, fold + 1)
        best_val_f1 = trainer.train()

        # ── FIX: Load Best Checkpoint Safely ─────────────────
        # PyTorch 2.6+ defaults weights_only=True, which blocks any non-tensor
        # type inside the checkpoint dict (e.g. numpy arrays, custom objects).
        #
        # Strategy: try weights_only=True first (safe, preferred).
        # If PyTorch still finds an unallowlisted type, fall back to
        # weights_only=False — safe here because WE created this checkpoint
        # during training in the same session (trusted source).
        try:
            checkpoint = torch.load(trainer.save_path, weights_only=True)
        except Exception:
            # Fallback: weights_only=False loads all Python objects freely.
            # This is acceptable ONLY for self-generated checkpoints.
            checkpoint = torch.load(trainer.save_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])

        # ── Evaluate on Test Subject ──────────────────────────
        model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                output = model(batch_x)
                pred = output.argmax(dim=1)
                all_preds.extend(pred.cpu().numpy())
                all_targets.extend(batch_y.cpu().numpy())

        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        acc = accuracy_score(all_targets, all_preds) * 100
        f1 = f1_score(all_targets, all_preds, average='macro')
        cm = confusion_matrix(all_targets, all_preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

        print(f"\n📊 Test Results - Subject {test_subj}:")
        print(f"  Accuracy: {acc:.2f}%")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Per-class:")
        print(f"    Neutral: {per_class_acc[0]:.1f}%")
        print(f"    Sad:     {per_class_acc[1]:.1f}%")
        print(f"    Fear:    {per_class_acc[2]:.1f}%")
        print(f"    Happy:   {per_class_acc[3]:.1f}%")
        print(f"  Confusion Matrix:\n{cm}")

        results.append({
            'subject': int(test_subj),
            'accuracy': acc,
            'f1': f1,
            'balance_ratio': balance_ratio
        })
        per_class_results.append({
            'subject': int(test_subj),
            'neutral': per_class_acc[0],
            'sad': per_class_acc[1],
            'fear': per_class_acc[2],
            'happy': per_class_acc[3]
        })
        confusion_matrices.append(cm)

        # Free GPU memory before next fold
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Final Summary ─────────────────────────────────────────
    if len(results) > 0:
        print("\n" + "=" * 60)
        print("FINAL BALANCED LOSO RESULTS")
        print("=" * 60)

        accs = [r['accuracy'] for r in results]
        f1s = [r['f1'] for r in results]

        print("\nPer-subject results:")
        for r in results:
            print(f"  Subject {r['subject']:2d}: Acc={r['accuracy']:.2f}%, F1={r['f1']:.4f}")

        print(f"\n📈 Overall Performance:")
        print(f"  Mean Accuracy: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")
        print(f"  Mean F1 Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

        neutral_accs = [p['neutral'] for p in per_class_results]
        sad_accs = [p['sad'] for p in per_class_results]
        fear_accs = [p['fear'] for p in per_class_results]
        happy_accs = [p['happy'] for p in per_class_results]

        print(f"\n📊 Per-class accuracy (mean ± std):")
        print(f"  Neutral: {np.mean(neutral_accs):.2f}% ± {np.std(neutral_accs):.2f}%")
        print(f"  Sad:     {np.mean(sad_accs):.2f}% ± {np.std(sad_accs):.2f}%")
        print(f"  Fear:    {np.mean(fear_accs):.2f}% ± {np.std(fear_accs):.2f}%")
        print(f"  Happy:   {np.mean(happy_accs):.2f}% ± {np.std(happy_accs):.2f}%")

        # Balance Quality: measures how evenly distributed per-class performance is
        # Perfect score = 100, lower = some classes are much worse than others
        per_class_means = [np.mean(neutral_accs), np.mean(sad_accs),
                           np.mean(fear_accs), np.mean(happy_accs)]
        balance_quality = 100 - np.std(per_class_means) * 2
        print(f"\n⚖️  Balance Quality Score: {balance_quality:.1f}/100")

        best_idx = np.argmax(accs)
        worst_idx = np.argmin(accs)
        print(f"\n  Best Subject:  {results[best_idx]['subject']} ({accs[best_idx]:.2f}%)")
        print(f"  Worst Subject: {results[worst_idx]['subject']} ({accs[worst_idx]:.2f}%)")

        avg_cm = np.mean(confusion_matrices, axis=0)
        print(f"\n📉 Average Confusion Matrix:")
        print(avg_cm.round(1))

        print("=" * 60)

    return results, per_class_results


# ─── CELL 13: MAIN EXECUTION ─────────────────────────────────

if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("BALANCED SOGNN FOR SEED-IV EMOTION RECOGNITION")
    print("WITH CLASS BALANCE OPTIMIZATION")
    print("=" * 60)

    set_seed(Config.SEED)
    print(f"\nUsing device: {device}")

    # ── Step 1: Load and Preprocess Data ─────────────────────
    print("\n" + "-" * 40)
    print("STEP 1: DATA LOADING")
    print("-" * 40)
    preprocessor = SEEDIVDataPreprocessor(Config)
    data_list, labels, subjects = preprocessor.load_data()
    data = preprocessor.standardize_time(data_list)
    data = preprocessor.normalize_per_subject(data, subjects)

    # ── Step 2: Feature Extraction ────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 2: FEATURE EXTRACTION")
    print("-" * 40)
    extractor = MultiFeatureExtractor(Config)
    data = extractor.extract_features(data)

    print(f"\nFinal data shape: {data.shape}")
    print(f"Final labels distribution: {np.bincount(labels)}")

    # ── Step 3: LOSO Cross-Validation ────────────────────────
    print("\n" + "-" * 40)
    print("STEP 3: BALANCED LOSO CROSS-VALIDATION")
    print("-" * 40)
    results, per_class_results = run_balanced_loso(data, labels, subjects, Config, device)

    # ── Step 4: Save Results ──────────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 4: SAVING RESULTS")
    print("-" * 40)

    results_file = os.path.join(Config.RESULTS_PATH, 'balanced_results.pkl')
    with open(results_file, 'wb') as f:
        pickle.dump({
            'results': results,
            'per_class': per_class_results,
            'config': {k: v for k, v in vars(Config).items()
                        if not k.startswith('__') and not callable(v)}
            # NOTE: Config.__dict__ returns a mappingproxy (not picklable).
            # We convert it to a plain dict, skipping dunder attrs and methods.
        }, f)

    print(f"✓ Results saved to {results_file}")

    print("\n" + "=" * 60)
    print("EXPERIMENT COMPLETE")
    print("=" * 60)

✓ Seed set to 42
BALANCE-OPTIMIZED CONFIGURATION
Device: cpu
Class weights: [0.8, 1.0, 0.7, 1.2]
Batch size: 128 (32 per class)
Focal loss: gamma=2.5, alpha=[0.2, 0.25, 0.2, 0.35]
Label smoothing: 0.1

BALANCED SOGNN FOR SEED-IV EMOTION RECOGNITION
WITH CLASS BALANCE OPTIMIZATION
✓ Seed set to 42

Using device: cpu

----------------------------------------
STEP 1: DATA LOADING
----------------------------------------

Loading 45 files...

✓ Loaded 1080 trials
  Time range: 10-64 points
  Labels: {np.int64(0): np.int64(270), np.int64(1): np.int64(270), np.int64(2): np.int64(270), np.int64(3): np.int64(270)}
  Balance ratio (max/min): 1.00

⏱️ Time standardized to 35: (1080, 62, 35, 5)

📊 Normalized: mean=-0.0000, std=1.0000

----------------------------------------
STEP 2: FEATURE EXTRACTION
----------------------------------------

✓ Multi-features extracted: (1080, 62, 35, 20)

Final data shape: (1080, 62, 35, 20)
Final labels distribution: [270 270 270 270]

-------------------------